# A3-mix: Decorrelated Rotation-Value Model

A submission variant combining market-state interactions, decorrelated price/liquidity features, industry-relative PIT fundamentals, and true SIZE exposure. Raw date-level states are not model inputs.


In [1]:
def main(datasources, start_date, end_date):
    """
    A3-mix: a decorrelated A3 variant that combines three changes:
    1) market states enter only through stock-specific interactions;
    2) overlapping price, volatility, and liquidity features are pruned;
    3) PIT fundamentals use industry-relative levels and four-disclosure changes.

    The training interval is fixed. Platform-provided dates are used only for
    out-of-sample prediction. The returned columns are date, instrument, factor.
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog
    import xgboost as xgb

    logger = structlog.get_logger()

    TRAIN_START = "2022-01-01 00:00:00"
    TRAIN_END = "2023-12-31 23:59:59"
    VALIDATION_START = pd.Timestamp("2023-01-01")
    TRAIN_BAR_TABLE = "bigalpha_2026_stock_bar1m"
    TRAIN_FINANCIAL_TABLE = "bigalpha_2026_financial"
    POOL_TABLE = "bigalpha_2026_instruments"
    EXPOSURE_TABLE = "bigalpha_2026_exposure"
    MIN_DAILY_COVERAGE = 0.60
    MIN_FINANCIAL_COVERAGE = 0.30
    MIN_EXPOSURE_COVERAGE = 0.60

    financial_cols = [
        "eps_basic",
        "operating_revenue",
        "net_profit_to_parent_shareholders",
        "total_assets",
        "total_equity_to_parent_shareholders",
    ]
    industry_cols = [
        "AERODEF",
        "AGRIFOREST",
        "AUTO",
        "BANK",
        "BEAUTY",
        "BUILDDECO",
        "CHEM",
        "COAL",
        "COMMETRADE",
        "COMPUTER",
        "CONGLOMERATES",
        "CONMAT",
        "ELECEQP",
        "ELECTRONICS",
        "ENVP",
        "FOODBEVER",
        "HEALTH",
        "HOUSEAPP",
        "IRONSTEEL",
        "LEISERVICE",
        "LIGHTINDUS",
        "MACHIEQUIP",
        "MEDIA",
        "MINING",
        "NONBANKFINAN",
        "NONFERMETAL",
        "PETRO",
        "REALESTATE",
        "TELECOM",
        "TEXTILE",
        "TRANSPORTATION",
        "UTILITIES",
    ]

    # The A3 duplicates ret_1, volatility_5, and liquidity_20 are deliberately
    # excluded. The two momentum legs use disjoint return windows.
    price_rank_cols = [
        "gap",
        "intraday_ret",
        "range_pct",
        "close_position",
        "momentum_5",
        "momentum_6_20",
        "downside_risk_20",
        "illiquidity_20",
        "volume_shock_20",
    ]
    financial_rank_cols = [
        "earnings_yield",
        "equity_ratio",
        "revenue_yoy",
        "profit_yoy",
        "roe_change_yoy",
    ]
    interaction_feature_cols = [
        "value_quality_score",
        "rotation_tilt",
        "defensive_quality",
        "trend_confirmation",
    ]
    feature_cols = (
        [f"rank_{col}" for col in price_rank_cols]
        + [f"industry_rank_{col}" for col in financial_rank_cols]
        + interaction_feature_cols
    )

    test_start = pd.to_datetime(start_date).normalize()
    test_end = pd.to_datetime(end_date).normalize()
    train_end = pd.to_datetime(TRAIN_END).normalize()
    if test_end < test_start:
        raise ValueError("end_date must not be earlier than start_date")
    if test_start <= train_end:
        raise ValueError("test interval overlaps the fixed training interval")
    if "bar1m" not in datasources or "financial" not in datasources:
        raise KeyError("A3-mix requires bar1m and financial data sources")

    def safe_ratio(numerator, denominator):
        numerator = pd.to_numeric(numerator, errors="coerce")
        denominator = pd.to_numeric(denominator, errors="coerce")
        denominator = denominator.where(denominator.abs() > 1e-12)
        return numerator.div(denominator)

    def safe_change(current, previous):
        # Scaling by abs(previous) keeps the direction interpretable when the
        # prior profit is negative and prevents division by tiny denominators.
        return safe_ratio(current - previous, previous.abs())

    def load_pool(sd, ed):
        pool = dai.query(
            f"SELECT date, instrument FROM {POOL_TABLE}",
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        if pool.empty:
            raise RuntimeError(f"empty stock pool for {sd} to {ed}")
        pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
        pool["instrument"] = pool["instrument"].astype(str)
        return (
            pool.drop_duplicates(["date", "instrument"])
                .sort_values(["date", "instrument"])
                .reset_index(drop=True)
        )

    def validate_daily_coverage(frame, pool, sd, ed, stage):
        sd_ts = pd.to_datetime(sd).normalize()
        ed_ts = pd.to_datetime(ed).normalize()
        target_pool = pool[
            (pool["date"] >= sd_ts) & (pool["date"] <= ed_ts)
        ]
        expected = target_pool.groupby("date")["instrument"].nunique()
        if expected.empty:
            raise RuntimeError(f"{stage}: no expected market dates")

        actual = frame.groupby("date")["instrument"].nunique()
        actual = actual.reindex(expected.index, fill_value=0)
        missing_dates = actual[actual == 0].index
        if len(missing_dates):
            missing_text = ", ".join(
                date.strftime("%Y-%m-%d") for date in missing_dates[:10]
            )
            raise RuntimeError(
                f"{stage}: missing complete market dates: {missing_text}"
            )

        coverage = actual.div(expected)
        low_coverage = coverage[coverage < MIN_DAILY_COVERAGE]
        if len(low_coverage):
            first_date = low_coverage.index[0].strftime("%Y-%m-%d")
            raise RuntimeError(
                f"{stage}: daily coverage below {MIN_DAILY_COVERAGE:.0%} "
                f"on {first_date}: {low_coverage.iloc[0]:.2%}"
            )

        return {
            "dates": int(len(expected)),
            "min_count": int(actual.min()),
            "median_count": float(actual.median()),
            "min_coverage": float(coverage.min()),
        }

    def load_daily_bars(bar_table, sd, ed):
        sql = f"""
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            instrument,
            ARG_MIN(open, date) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            ARG_MAX(close, date) AS close,
            ARG_MAX(pre_close, date) AS pre_close,
            ARG_MAX(volume, date) AS volume,
            ARG_MAX(amount, date) AS amount
        FROM {bar_table}
        GROUP BY trading_day, instrument
        ORDER BY trading_day, instrument
        """
        bars = dai.query(
            sql,
            filters={"date": [sd, ed]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})
        if bars.empty:
            raise RuntimeError(f"empty bar data for {sd} to {ed}")

        bars["date"] = pd.to_datetime(bars["date"]).dt.normalize()
        bars["instrument"] = bars["instrument"].astype(str)
        for col in [
            "open", "high", "low", "close", "pre_close", "volume", "amount"
        ]:
            bars[col] = pd.to_numeric(bars[col], errors="coerce")
            bars[col] = bars[col].replace([np.inf, -np.inf], np.nan)
            bars.loc[bars[col] <= 0, col] = np.nan
        return (
            bars.sort_values(["instrument", "date"])
                .drop_duplicates(["date", "instrument"])
                .reset_index(drop=True)
        )

    def load_financials(financial_table, sd, ed):
        query_start = pd.to_datetime(sd).normalize() - pd.Timedelta(days=800)
        select_cols = ", ".join(financial_cols)
        sql = f"""
        SELECT date, instrument, {select_cols}
        FROM {financial_table}
        WHERE category = 'lf' AND shift = 0
        ORDER BY date, instrument
        """
        financial = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df()
        if financial.empty:
            raise RuntimeError(f"empty financial data for {query_start} to {ed}")

        financial["date"] = pd.to_datetime(financial["date"]).dt.normalize()
        financial["instrument"] = financial["instrument"].astype(str)
        for col in financial_cols:
            financial[col] = pd.to_numeric(financial[col], errors="coerce")
            financial[col] = financial[col].replace(
                [np.inf, -np.inf], np.nan
            )
        financial = (
            financial.drop_duplicates(["date", "instrument"], keep="last")
                     .sort_values(["instrument", "date"])
                     .reset_index(drop=True)
        )

        # Four disclosure events approximate a same-quarter prior-year
        # comparison without relying on undocumented competition-only columns.
        grouped = financial.groupby("instrument", sort=False)
        prior = {}
        for col in financial_cols:
            prior[col] = grouped[col].shift(4)
        financial["revenue_yoy"] = safe_change(
            financial["operating_revenue"], prior["operating_revenue"]
        )
        financial["profit_yoy"] = safe_change(
            financial["net_profit_to_parent_shareholders"],
            prior["net_profit_to_parent_shareholders"],
        )
        current_roe = safe_ratio(
            financial["net_profit_to_parent_shareholders"],
            financial["total_equity_to_parent_shareholders"],
        )
        prior_roe = safe_ratio(
            prior["net_profit_to_parent_shareholders"],
            prior["total_equity_to_parent_shareholders"],
        )
        financial["roe_change_yoy"] = current_roe - prior_roe

        # Use a conservative next-calendar-day availability rule.
        financial["date"] = financial["date"] + pd.Timedelta(days=1)
        return financial.sort_values(["date", "instrument"]).reset_index(drop=True)

    def load_exposure(sd, ed):
        select_cols = ", ".join(["SIZE"] + industry_cols)
        sql = f"""
        SELECT date, instrument, {select_cols}
        FROM {EXPOSURE_TABLE}
        ORDER BY date, instrument
        """
        exposure = dai.query(
            sql,
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        if exposure.empty:
            raise RuntimeError(f"empty exposure data for {sd} to {ed}")
        exposure["date"] = pd.to_datetime(exposure["date"]).dt.normalize()
        exposure["instrument"] = exposure["instrument"].astype(str)
        exposure["SIZE"] = pd.to_numeric(exposure["SIZE"], errors="coerce")
        for col in industry_cols:
            exposure[col] = pd.to_numeric(exposure[col], errors="coerce").fillna(0)
        industry_values = exposure[industry_cols]
        exposure["industry"] = industry_values.idxmax(axis=1)
        exposure.loc[industry_values.max(axis=1) <= 0, "industry"] = "OTHER"
        return (
            exposure[["date", "instrument", "SIZE", "industry"]]
                .drop_duplicates(["date", "instrument"])
                .sort_values(["date", "instrument"])
                .reset_index(drop=True)
        )

    def add_market_interactions(data):
        size_pct = data.groupby("date")["SIZE"].rank(
            method="average", pct=True
        )
        data["rank_size_exposure"] = size_pct - 0.5
        data["large_return_component"] = data["ret_1"].where(size_pct >= 0.70)
        data["small_return_component"] = data["ret_1"].where(size_pct <= 0.30)

        market = (
            data.groupby("date", as_index=False)
                .agg(
                    market_return=("ret_1", "mean"),
                    positive_share=(
                        "ret_1",
                        lambda s: float((s.dropna() > 0).mean()),
                    ),
                    large_return=("large_return_component", "mean"),
                    small_return=("small_return_component", "mean"),
                )
                .sort_values("date")
                .reset_index(drop=True)
        )
        market["market_log_return"] = np.log1p(
            market["market_return"].clip(lower=-0.999999)
        )
        market["market_trend_20"] = np.expm1(
            market["market_log_return"].rolling(20, min_periods=10).sum()
        )
        market["market_volatility_20"] = market["market_return"].rolling(
            20, min_periods=10
        ).std()
        market["market_breadth_20"] = (
            market["positive_share"].rolling(20, min_periods=10).mean() - 0.5
        )
        market["size_rotation_20"] = (
            market["large_return"] - market["small_return"]
        ).rolling(20, min_periods=10).sum()
        scaled_trend = safe_ratio(
            market["market_trend_20"],
            market["market_volatility_20"] * np.sqrt(20.0),
        )
        market["risk_on_state"] = np.tanh(scaled_trend.clip(-3.0, 3.0))
        state_cols = [
            "date",
            "market_breadth_20",
            "size_rotation_20",
            "risk_on_state",
        ]
        data = pd.merge(
            data,
            market[state_cols],
            on="date",
            how="left",
            validate="many_to_one",
        )

        value_parts = [
            "industry_rank_earnings_yield",
            "industry_rank_equity_ratio",
            "industry_rank_revenue_yoy",
            "industry_rank_profit_yoy",
            "industry_rank_roe_change_yoy",
        ]
        data["value_quality_score"] = data[value_parts].mean(
            axis=1, skipna=True
        )
        rotation_state = np.tanh(4.0 * data["size_rotation_20"].fillna(0.0))
        data["rotation_tilt"] = data["rank_size_exposure"] * rotation_state
        defensive_weight = (1.0 - data["risk_on_state"].fillna(0.0)) / 2.0
        breadth_guard = 1.0 - data["market_breadth_20"].fillna(0.0).clip(
            -0.5, 0.5
        )
        data["defensive_quality"] = (
            data["value_quality_score"] * defensive_weight * breadth_guard
        )
        data["trend_confirmation"] = (
            data["rank_momentum_6_20"]
            * data["risk_on_state"].fillna(0.0)
        )

        # Raw date-level states and the direct SIZE exposure do not enter the
        # model. Only these stock-specific interactions survive in feature_cols.
        return data.drop(
            columns=["large_return_component", "small_return_component"]
        )

    def build_dataset(bar_table, financial_table, sd, ed, need_label=False):
        t0 = time.time()
        sd_ts = pd.to_datetime(sd).normalize()
        ed_ts = pd.to_datetime(ed).normalize()
        query_start = sd_ts - pd.Timedelta(days=120)
        query_end = ed_ts + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

        bars = load_daily_bars(bar_table, query_start, query_end)
        bars["ret_1"] = bars["close"] / bars["pre_close"] - 1.0
        bars["gap"] = bars["open"] / bars["pre_close"] - 1.0
        bars["intraday_ret"] = bars["close"] / bars["open"] - 1.0
        bars["range_pct"] = (bars["high"] - bars["low"]) / bars["pre_close"]
        price_range = bars["high"] - bars["low"]
        bars["close_position"] = (
            (bars["close"] - bars["low"])
            / price_range.replace(0, np.nan)
            - 0.5
        )

        grouped = bars.groupby("instrument", sort=False)
        bars["log_return"] = np.log1p(bars["ret_1"].clip(lower=-0.999999))
        log_sum_5 = grouped["log_return"].transform(
            lambda s: s.rolling(5, min_periods=3).sum()
        )
        log_sum_20 = grouped["log_return"].transform(
            lambda s: s.rolling(20, min_periods=10).sum()
        )
        bars["momentum_5"] = np.expm1(log_sum_5)
        bars["momentum_6_20"] = np.expm1(log_sum_20 - log_sum_5)
        bars["negative_return_sq"] = bars["ret_1"].clip(upper=0.0).pow(2)
        bars["downside_risk_20"] = grouped["negative_return_sq"].transform(
            lambda s: np.sqrt(s.rolling(20, min_periods=10).mean())
        )
        bars["average_volume_20"] = grouped["volume"].transform(
            lambda s: s.rolling(20, min_periods=10).mean()
        )
        bars["volume_shock_20"] = np.log(
            safe_ratio(bars["volume"], bars["average_volume_20"])
        )
        bars["daily_illiquidity"] = safe_ratio(
            bars["ret_1"].abs() * 1e8, bars["amount"]
        )
        bars["illiquidity_20"] = grouped["daily_illiquidity"].transform(
            lambda s: np.log1p(s.rolling(20, min_periods=10).mean())
        )

        pool = load_pool(query_start, ed_ts)
        data = pd.merge(
            pool,
            bars,
            on=["date", "instrument"],
            how="inner",
            validate="one_to_one",
        )
        financial = load_financials(financial_table, query_start, ed_ts)
        data = pd.merge_asof(
            data.sort_values(["date", "instrument"]),
            financial.sort_values(["date", "instrument"]),
            on="date",
            by="instrument",
            direction="backward",
            allow_exact_matches=True,
        )
        exposure = load_exposure(query_start, ed_ts)
        data = pd.merge(
            data,
            exposure,
            on=["date", "instrument"],
            how="left",
            validate="one_to_one",
        )

        data["earnings_yield"] = safe_ratio(data["eps_basic"], data["close"])
        data["equity_ratio"] = safe_ratio(
            data["total_equity_to_parent_shareholders"], data["total_assets"]
        )

        for col in price_rank_cols:
            data[col] = pd.to_numeric(data[col], errors="coerce")
            data[col] = data[col].replace([np.inf, -np.inf], np.nan)
            data[f"rank_{col}"] = (
                data.groupby("date")[col]
                    .rank(method="average", pct=True)
                    .sub(0.5)
            )
        for col in financial_rank_cols:
            data[col] = pd.to_numeric(data[col], errors="coerce")
            data[col] = data[col].replace([np.inf, -np.inf], np.nan)
            data[f"industry_rank_{col}"] = (
                data.groupby(["date", "industry"])[col]
                    .rank(method="average", pct=True)
                    .sub(0.5)
            )

        target_rows = data[
            (data["date"] >= sd_ts) & (data["date"] <= ed_ts)
        ]
        financial_coverage = float(
            target_rows["industry_rank_equity_ratio"].notna().mean()
        )
        exposure_coverage = float(target_rows["SIZE"].notna().mean())
        if exposure_coverage < MIN_EXPOSURE_COVERAGE:
            raise RuntimeError(
                "A3-mix exposure coverage below "
                f"{MIN_EXPOSURE_COVERAGE:.0%}: {exposure_coverage:.2%}"
            )
        if financial_coverage < MIN_FINANCIAL_COVERAGE:
            raise RuntimeError(
                "A3-mix financial feature coverage below "
                f"{MIN_FINANCIAL_COVERAGE:.0%}: {financial_coverage:.2%}"
            )

        data = add_market_interactions(data)

        if need_label:
            market_dates = pd.Series(
                pool["date"].drop_duplicates().sort_values().to_numpy()
            )
            next_date_map = pd.DataFrame({
                "date": market_dates.iloc[:-1].to_numpy(),
                "next_date": market_dates.iloc[1:].to_numpy(),
            })
            next_returns = bars[["date", "instrument", "ret_1"]].rename(
                columns={"date": "next_date", "ret_1": "future_return_1d"}
            )
            data = pd.merge(data, next_date_map, on="date", how="left")
            data = pd.merge(
                data,
                next_returns,
                on=["next_date", "instrument"],
                how="left",
                validate="many_to_one",
            )
            data["label"] = (
                data.groupby("date")["future_return_1d"]
                    .rank(method="average", pct=True)
                    .sub(0.5)
            )

        data = data[
            (data["date"] >= sd_ts) & (data["date"] <= ed_ts)
        ].copy()
        daily_checks = validate_daily_coverage(
            data, pool, sd_ts, ed_ts, stage="A3-mix feature data"
        )

        if need_label:
            target_pool = pool[
                (pool["date"] >= sd_ts) & (pool["date"] <= ed_ts)
            ]
            expected_labels = (
                target_pool.groupby("date")["instrument"].nunique().iloc[:-1]
            )
            actual_labels = (
                data.dropna(subset=["label"])
                    .groupby("date")["instrument"]
                    .nunique()
                    .reindex(expected_labels.index, fill_value=0)
            )
            label_coverage = actual_labels.div(expected_labels)
            low_label_coverage = label_coverage[
                label_coverage < MIN_DAILY_COVERAGE
            ]
            if len(low_label_coverage):
                first_date = low_label_coverage.index[0].strftime("%Y-%m-%d")
                raise RuntimeError(
                    "A3-mix labels: insufficient strict T+1 labels on "
                    f"{first_date}: {low_label_coverage.iloc[0]:.2%}"
                )

        missing_rates = data[feature_cols].isna().mean().round(4).to_dict()
        logger.info(
            "A3-mix build_dataset complete",
            bar_table=bar_table,
            financial_table=financial_table,
            rows=len(data),
            dates=data["date"].nunique(),
            financial_coverage=round(financial_coverage, 4),
            exposure_coverage=round(exposure_coverage, 4),
            daily_checks=daily_checks,
            missing_rates=missing_rates,
            elapsed=round(time.time() - t0, 2),
        )
        return data.reset_index(drop=True)

    def make_model():
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=72,
            max_depth=3,
            learning_rate=0.035,
            min_child_weight=120,
            subsample=0.80,
            colsample_bytree=0.80,
            max_bin=64,
            reg_alpha=0.30,
            reg_lambda=10.0,
            tree_method="hist",
            n_jobs=4,
            random_state=42,
            verbosity=0,
        )

    logger.info("building A3-mix training data")
    train_df = build_dataset(
        TRAIN_BAR_TABLE,
        TRAIN_FINANCIAL_TABLE,
        TRAIN_START,
        TRAIN_END,
        need_label=True,
    )
    train_df = (
        train_df.dropna(subset=["label"])
                .dropna(subset=feature_cols, how="all")
                .reset_index(drop=True)
    )
    if train_df.empty or train_df["date"].nunique() < 200:
        raise RuntimeError("insufficient A3-mix training data")

    development = train_df[train_df["date"] < VALIDATION_START].copy()
    validation = train_df[train_df["date"] >= VALIDATION_START].copy()
    if development.empty or validation.empty:
        raise RuntimeError("A3-mix temporal validation split is empty")

    validation_model = make_model()
    validation_model.fit(development[feature_cols], development["label"])
    validation["prediction"] = validation_model.predict(
        validation[feature_cols]
    )
    validation["prediction_rank"] = (
        validation.groupby("date")["prediction"]
                  .rank(method="average", pct=True)
                  .sub(0.5)
    )
    daily_ic = validation.groupby("date").apply(
        lambda frame: frame["prediction_rank"].corr(frame["label"])
    ).dropna()
    ic_std = float(daily_ic.std(ddof=1)) if len(daily_ic) > 1 else np.nan
    validation_metrics = {
        "dates": int(len(daily_ic)),
        "ic_mean": float(daily_ic.mean()) if len(daily_ic) else np.nan,
        "ic_ir": (
            float(daily_ic.mean() / ic_std)
            if np.isfinite(ic_std) and ic_std > 0
            else np.nan
        ),
        "positive_ic_rate": (
            float((daily_ic > 0).mean()) if len(daily_ic) else np.nan
        ),
    }
    logger.info(
        "A3-mix 2023 temporal validation",
        metrics={
            key: round(value, 6) if isinstance(value, float) else value
            for key, value in validation_metrics.items()
        },
    )

    model = make_model()
    fit_start = time.time()
    model.fit(train_df[feature_cols], train_df["label"])
    importance = pd.Series(
        model.feature_importances_, index=feature_cols
    ).sort_values(ascending=False)
    logger.info(
        "A3-mix final model trained",
        rows=len(train_df),
        dates=train_df["date"].nunique(),
        elapsed=round(time.time() - fit_start, 2),
        importance=importance.round(4).to_dict(),
    )

    test_df = build_dataset(
        datasources["bar1m"],
        datasources["financial"],
        start_date,
        end_date,
        need_label=False,
    ).dropna(subset=feature_cols, how="all")
    if test_df.empty:
        raise RuntimeError("empty A3-mix test data after feature checks")

    test_df["prediction"] = model.predict(test_df[feature_cols])
    test_df["factor"] = (
        test_df.groupby("date")["prediction"]
               .rank(method="average", pct=True)
               .sub(0.5)
    )

    result = test_df[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result = result[np.isfinite(result["factor"])].copy()
    result = (
        result.drop_duplicates(["date", "instrument"])
              .sort_values(["date", "instrument"])
              .reset_index(drop=True)
    )
    if result.empty or result.duplicated(["date", "instrument"]).any():
        raise RuntimeError("invalid A3-mix factor output")

    test_pool = load_pool(start_date, end_date)
    daily_checks = validate_daily_coverage(
        result, test_pool, start_date, end_date, stage="A3-mix factor output"
    )
    logger.info(
        "A3-mix factor complete",
        rows=len(result),
        dates=result["date"].nunique(),
        daily_checks=daily_checks,
    )
    return result


if __name__ == "__main__":
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    factor_data = main(datasources, start_date, end_date)
    print("A3-mix factor shape:", factor_data.shape)
    print(
        "A3-mix date range:",
        factor_data["date"].min(),
        factor_data["date"].max(),
    )
    print(factor_data.head())


[2026-07-21 11:23:22] [info     ] building A3-mix training data
[2026-07-21 11:25:19] [info     ] A3-mix build_dataset complete  bar_table=bigalpha_2026_stock_bar1m daily_checks={'dates': 484, 'min_count': 992, 'median_count': 998.0, 'min_coverage': 0.992} dates=484 elapsed=117.19 exposure_coverage=1.0 financial_coverage=1.0 financial_table=bigalpha_2026_financial missing_rates={'rank_gap': 0.0034, 'rank_intraday_ret': 0.0034, 'rank_range_pct': 0.0034, 'rank_close_position': 0.004, 'rank_momentum_5': 0.0035, 'rank_momentum_6_20': 0.0038, 'rank_downside_risk_20': 0.0035, 'rank_illiquidity_20': 0.0059, 'rank_volume_shock_20': 0.0038, 'industry_rank_earnings_yield': 0.0044, 'industry_rank_equity_ratio': 0.0, 'industry_rank_revenue_yoy': 0.0009, 'industry_rank_profit_yoy': 0.0008, 'industry_rank_roe_change_yoy': 0.0022, 'value_quality_score': 0.0, 'rotation_tilt': 0.0, 'defensive_quality': 0.0, 'trend_confirmation': 0.0038} rows=483141
[2026-07-21 11:25:30] [info     ] A3-mix 2023 temporal